In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os
from PIL import Image
from tqdm.notebook import tqdm
import torch
from topodataset import TopoDataSet
import utils


In [ ]:
os.getcwd()

In [ ]:
# load the data:
# DATASETS/DATALOADER
bs=8
num_workers=0
downsampling_factor=1
limit_num_data=np.inf
crop_size=300
threshold_input=True


dataset = TopoDataSet(
    dir='../data/',
    downsampling_factor=downsampling_factor,
    limit_num_data=limit_num_data,
    crop_size=crop_size,
    threshold_input=threshold_input,
    return_filename=True
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=bs,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)


In [ ]:
source = 'real'
stain_names = ['background', 'YAP', 'Actin', 'DAPI']
save_folder = 'real_images_29-4-25_insert_preproc'


In [ ]:
counter = 0

for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    for (img, bg, fname) in zip(*batch):
        obj_num = fname.split('Nr')[1].split('.png')[0]
        chip = fname.split('/')[1].split('\\')[0]
        save_fold_this = os.path.join(save_folder, chip +f'_{source}')
        if not os.path.exists(save_fold_this):
            os.makedirs(save_fold_this)
        all_channels = torch.cat([bg, img], dim=0)
        # print(all_channels.shape)
        # to_plot = utils.plot_topo_cells_one_im(all_channels.permute(1,2,0).numpy(), bg_norm=0.4)
        # plt.imshow(to_plot)
        # plt.show()
        # normalize to [0,1] instead of [-1,1]:
        all_channels = (all_channels + 1) / 2
        assert (0 <= all_channels.min()) and (all_channels.max() <= 1)
        for c, channel in enumerate(all_channels):
            assert channel.shape == (crop_size, crop_size)
            try:
                im_channel = Image.fromarray((channel.numpy() * 255).astype(np.uint8))
                fname_save = f'obj{obj_num}_stain{stain_names[c]}_{source}.tiff'
                # print(fname, save_fold_this, fname_save)
                im_channel.save(os.path.join(save_fold_this, fname_save))
                counter += 1
            except Exception as e:
                print(f'failed to convert {fname}')
                print(e)




In [ ]:
fname

In [ ]:
test = Image.open(
    os.path.join(save_folder, 'Chip_1130_1_real/obj0041_stainDAPI_real.tiff')
    )
plt.hist(np.array(test).reshape(-1), bins=100)
plt.yscale('log')
plt.show()

plt.imshow(np.array(test))
plt.show()